# 04 — Mitigações: aumento forte, AdaBN e fine-tuning

Parte dos modelos base do `02_treino.ipynb` e testa três formas de reduzir a queda no PlantDoc,
de forma **acumulada**, para os três modelos:

| Variante | O que muda | Usa PlantDoc no treino? |
|---|---|---|
| `base` | modelo do notebook 02 | não |
| `base+adabn` | recalcula as estatísticas do BatchNorm com fotos de campo | só imagens, **sem rótulos** |
| `aug` | treino do zero com aumento forte + CutMix + label smoothing | não |
| `aug+adabn` | `aug` + AdaBN | só imagens, **sem rótulos** |
| `aug+ft` | `aug` + fine-tuning com fotos de campo rotuladas | sim (parte de treino) |

**Regras para não "roubar":**
- O **teste oficial do PlantDoc (236 fotos) nunca é usado** para treinar, adaptar ou escolher modelos.
- AdaBN usa as imagens da parte de **treino** do PlantDoc, sem rótulos.
- O fine-tuning usa 85% da parte de treino do PlantDoc (+ o mesmo número de imagens do PlantVillage,
  para o modelo não esquecer); os outros 15% servem de validação para escolher a melhor época.

**Como usar:** igual ao 02. Rode com `QUICK = True` (~10 min) para pegar erros; depois `QUICK = False` (~45 min).
Se o Colab cair, rode de novo: o que já terminou é pulado.

**Saídas:** modelos em `MyDrive/leafscan/runs/<run>_aug`, `<run>_aug_ft`, `<run>_adabn`, `<run>_aug_adabn`;
tabela em `results/tables/tab_mitigacao.csv` e `.tex`; figura `results/figs/mitigacao.png`.

## 1. Configuração

In [ ]:
QUICK = True   # True = teste rápido; False = execução completa

from google.colab import drive
drive.mount('/content/drive')

import os, json, time, math, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

DRIVE_DIR = '/content/drive/MyDrive/leafscan'
CACHE_DIR = f'{DRIVE_DIR}/cache'
BASE_DIR  = f'{DRIVE_DIR}/runs'                                  # modelos base (notebook 02)
OUT_DIR   = f'{DRIVE_DIR}/runs_quick' if QUICK else f'{DRIVE_DIR}/runs'
TAB_DIR   = f'{DRIVE_DIR}/results/tables'
FIG_DIR   = f'{DRIVE_DIR}/results/figs'
LOCAL     = '/content/cache'

SEED      = 42
RUNS      = ['cnn_base', 'mnv3s', 'mnv2_050']
# treino do zero com aumento forte (mesmo orçamento dos modelos base)
EPOCHS_AUG, BATCH, LR, WD, PATIENCE = (2 if QUICK else 20), 128, 2e-3, 1e-4, 6
# fine-tuning
EPOCHS_FT, BATCH_FT, LR_FT, PATIENCE_FT = (2 if QUICK else 15), 64, 5e-4, 5
LABEL_SMOOTHING = 0.1

for d in (OUT_DIR, TAB_DIR, FIG_DIR, LOCAL):
    os.makedirs(d, exist_ok=True)
for r in RUNS:
    assert os.path.exists(f'{BASE_DIR}/{r}/best.keras'), f'Falta o modelo base {r}: rode o notebook 02'

keras.utils.set_random_seed(SEED)
keras.mixed_precision.set_global_policy('mixed_float16')
print('TF', tf.__version__, '| Keras', keras.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Dados

In [ ]:
for name in ('pv_x', 'pv_y', 'pv_split', 'pd_x', 'pd_y', 'pd_split'):
    src, dst = f'{CACHE_DIR}/{name}.npy', f'{LOCAL}/{name}.npy'
    # copia de novo se não existir ou se uma cópia anterior ficou incompleta
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        shutil.copy(src, dst + '.tmp'); os.replace(dst + '.tmp', dst)
meta = json.load(open(f'{CACHE_DIR}/meta.json'))
PV_CLASSES = meta['pv_classes']; NUM_CLASSES = len(PV_CLASSES)
PD_IDX = np.array(meta['pd_pv_indices']); IMG = meta['img_size']
CROP = np.array([c.split('___')[0] for c in PV_CLASSES])

pv_x = np.load(f'{LOCAL}/pv_x.npy'); pv_y = np.load(f'{LOCAL}/pv_y.npy').astype(np.int32)
pv_split = np.load(f'{LOCAL}/pv_split.npy')
pd_x = np.load(f'{LOCAL}/pd_x.npy'); pd_y = np.load(f'{LOCAL}/pd_y.npy').astype(np.int32)
pd_split = np.load(f'{LOCAL}/pd_split.npy')

x_tr, y_tr = pv_x[pv_split == 0], pv_y[pv_split == 0]
x_va, y_va = pv_x[pv_split == 1], pv_y[pv_split == 1]
x_te, y_te = pv_x[pv_split == 2], pv_y[pv_split == 2]
del pv_x
x_pdtr, y_pdtr = pd_x[pd_split == 0], pd_y[pd_split == 0]     # parte de treino do PlantDoc
x_pd,   y_pd   = pd_x[pd_split == 2], pd_y[pd_split == 2]     # teste oficial (só avaliação)

# fine-tuning: 85% treino / 15% validação dentro da parte de treino do PlantDoc
i_ft, i_ftva = train_test_split(np.arange(len(y_pdtr)), test_size=0.15, stratify=y_pdtr, random_state=SEED)
x_ftva, y_ftva = x_pdtr[i_ftva], y_pdtr[i_ftva]
rng = np.random.default_rng(SEED)
i_pvmix = rng.choice(len(y_tr), len(i_ft), replace=False)       # mesmo nº de imagens do PlantVillage
x_ft = np.concatenate([x_pdtr[i_ft], x_tr[i_pvmix]]); y_ft = np.concatenate([y_pdtr[i_ft], y_tr[i_pvmix]])

if QUICK:
    i = rng.choice(len(y_tr), len(y_tr) // 10, replace=False); x_tr, y_tr = x_tr[i], y_tr[i]
    i = rng.choice(len(y_va), len(y_va) // 10, replace=False); x_va, y_va = x_va[i], y_va[i]

print(f'PV treino {len(y_tr)} | PV val {len(y_va)} | PV teste {len(y_te)}')
print(f'PlantDoc: AdaBN {len(y_pdtr)} (sem rótulo) | FT treino {len(y_ft)} ({len(i_ft)} campo + {len(i_ft)} PV) '
      f'| FT val {len(y_ftva)} | teste {len(y_pd)}')

## 3. Aumento de dados forte + CutMix
Objetivo: tirar do modelo os atalhos do laboratório (enquadramento fixo, fundo liso, iluminação controlada).
- recorte aleatório agressivo (12–100% da área), 8 simetrias;
- brilho, contraste e saturação fortes; desfoque gaussiano (30%); ruído;
- **CutMix** (50% dos lotes): cola um retângulo de outra imagem e mistura os rótulos na proporção da área;
- **label smoothing** 0,1 na perda (também ajuda a calibrar a confiança, que era o problema do limiar do app).

In [ ]:
_g = np.exp(-0.5 * (np.arange(-2, 3) / 1.2) ** 2); _g = np.outer(_g, _g); _g /= _g.sum()
GAUSS = tf.constant(np.repeat(_g[:, :, None, None], 3, axis=2), tf.float32)   # [5,5,3,1]
GRAY_W = tf.constant([0.299, 0.587, 0.114], tf.float32)

def dihedral(x, b):
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)
    t = tf.random.uniform([b, 1, 1, 1]) < 0.5
    return tf.where(t, tf.transpose(x, [0, 2, 1, 3]), x)

def random_crop(x, b, lo):
    s = tf.random.uniform([b], lo, 1.0)
    oy = tf.random.uniform([b]) * (1 - s); ox = tf.random.uniform([b]) * (1 - s)
    return tf.image.crop_and_resize(x, tf.stack([oy, ox, oy + s, ox + s], 1), tf.range(b), [IMG, IMG])

def cutmix(x, y, b):
    perm = tf.random.shuffle(tf.range(b))
    r = tf.sqrt(tf.random.uniform([], 0.1, 0.6))           # lado do retângulo (fração)
    ch = tf.cast(r * IMG, tf.int32)
    cy = tf.random.uniform([], 0, IMG - ch + 1, tf.int32); cx = tf.random.uniform([], 0, IMG - ch + 1, tf.int32)
    rows = tf.range(IMG)[:, None]; cols = tf.range(IMG)[None, :]
    m = tf.cast((rows >= cy) & (rows < cy + ch) & (cols >= cx) & (cols < cx + ch), tf.float32)[None, :, :, None]
    area = tf.reduce_mean(m)
    return x * (1 - m) + tf.gather(x, perm) * m, y * (1 - area) + tf.gather(y, perm) * area

def augment_strong(x, y):
    b = tf.shape(x)[0]
    x = dihedral(x, b)
    x = random_crop(x, b, 0.35)
    x = x + tf.random.uniform([b, 1, 1, 1], -40.0, 40.0)
    m = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
    x = (x - m) * tf.random.uniform([b, 1, 1, 1], 0.6, 1.4) + m
    g = tf.reduce_sum(x * GRAY_W, -1, keepdims=True)
    x = g + (x - g) * tf.random.uniform([b, 1, 1, 1], 0.6, 1.4)
    xb = tf.nn.depthwise_conv2d(x, GAUSS, [1, 1, 1, 1], 'SAME')
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.3, xb, x)
    x = x + tf.random.normal(tf.shape(x)) * tf.random.uniform([b, 1, 1, 1], 0.0, 8.0)
    x = tf.clip_by_value(x, 0.0, 255.0)
    x, y = tf.cond(tf.random.uniform([]) < 0.5, lambda: cutmix(x, y, b), lambda: (x, y))
    return x, y

def make_ds(x, y, training, batch, aug_fn=None):
    with tf.device('/CPU:0'):
        X = tf.constant(x); Y = tf.constant(y)
    ds = tf.data.Dataset.range(len(y))
    if training:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch).map(
        lambda i: (tf.cast(tf.gather(X, i), tf.float32), tf.one_hot(tf.gather(Y, i), NUM_CLASSES)),
        num_parallel_calls=tf.data.AUTOTUNE)
    if aug_fn is not None:
        ds = ds.map(aug_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

import matplotlib.pyplot as plt
xb, yb = next(iter(make_ds(x_tr[:256], y_tr[:256], True, 16, augment_strong)))
fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
for k, a in enumerate(ax.ravel()):
    a.imshow(xb[k].numpy().astype('uint8')); a.axis('off')
plt.suptitle('Exemplos do aumento forte'); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/aug_forte_exemplos.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. Modelos (mesma definição do notebook 02)

In [ ]:
def conv_bn(x, f):
    x = layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def fix_bn_momentum(base, momentum=0.99):
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.momentum = momentum

def build_model(name):
    inp = keras.Input((IMG, IMG, 3), name='image')
    if name == 'cnn_base':
        x = layers.Rescaling(1 / 255.0)(inp)
        for f in (32, 64, 128, 256):
            x = conv_bn(x, f); x = conv_bn(x, f)
            x = layers.MaxPooling2D()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.3)(x)
    elif name == 'mnv3s':
        base = keras.applications.MobileNetV3Small(input_shape=(IMG, IMG, 3), include_top=False, weights=None,
                                                   pooling='avg', include_preprocessing=True)
        fix_bn_momentum(base)
        x = layers.Dropout(0.2)(base(inp))
    elif name == 'mnv2_050':
        x = layers.Rescaling(1 / 127.5, offset=-1.0)(inp)
        base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), alpha=0.5, include_top=False,
                                              weights=None, pooling='avg')
        fix_bn_momentum(base)
        x = layers.Dropout(0.2)(base(x))
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='probs')(x)
    return keras.Model(inp, out, name=name)

def bn_layers(m):
    out = []
    for l in m.layers:
        if isinstance(l, keras.Model):
            out += bn_layers(l)
        elif isinstance(l, layers.BatchNormalization):
            out.append(l)
    return out

## 5. Avaliação (mesmas métricas do 02 + cultura e calibração)

In [ ]:
def predict_probs(model, x):
    ds = make_ds(x, np.zeros(len(x), np.int32), training=False, batch=256)
    return model.predict(ds.map(lambda a, b: a), verbose=0)

def ece(p, y, bins=15):
    conf, pred = p.max(1), p.argmax(1)
    e = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        k = (conf > lo) & (conf <= lo + 1 / bins)
        if k.any():
            e += k.mean() * abs((pred[k] == y[k]).mean() - conf[k].mean())
    return e

def evaluate(model, run, variant):
    p_pv, p_pd = predict_probs(model, x_te), predict_probs(model, x_pd)
    pv, pdp = p_pv.argmax(1), p_pd.argmax(1)
    pd_r = PD_IDX[p_pd[:, PD_IDX].argmax(1)]
    k = p_pd.max(1) >= 0.7
    return {
        'run': run, 'variante': variant,
        'pv_acc': accuracy_score(y_te, pv), 'pv_f1': f1_score(y_te, pv, average='macro'),
        'pd_acc': accuracy_score(y_pd, pdp),
        'pd_f1': f1_score(y_pd, pdp, labels=PD_IDX, average='macro', zero_division=0),
        'pd_acc_restrito': accuracy_score(y_pd, pd_r),
        'pd_acc_cultura': float((CROP[y_pd] == CROP[pdp]).mean()),
        'pd_ece': ece(p_pd, y_pd),
        'pd_cob@0.7': float(k.mean()),
        'pd_acc@0.7': float((pdp[k] == y_pd[k]).mean()) if k.any() else float('nan'),
    }, p_pv, p_pd

def save_eval(model, run, variant, rdir):
    path = f'{rdir}/metrics_{variant}.json'
    if os.path.exists(path):
        return json.load(open(path))
    m, p_pv, p_pd = evaluate(model, run, variant)
    np.save(f'{rdir}/pred_pv_test_{variant}.npy', p_pv.astype(np.float16))
    np.save(f'{rdir}/pred_pd_test_{variant}.npy', p_pd.astype(np.float16))
    json.dump(m, open(path, 'w'), indent=2)
    print(f'  [{run} | {variant:10s}] PV {m["pv_acc"]:.3f} | PlantDoc {m["pd_acc"]:.3f} '
          f'(restr. {m["pd_acc_restrito"]:.3f}, cultura {m["pd_acc_cultura"]:.3f}, ECE {m["pd_ece"]:.3f})')
    return m

## 6. AdaBN
Passa as imagens de treino do PlantDoc (sem rótulos) pela rede em modo de treino e substitui as médias e
variâncias do BatchNorm pela **média exata** dessas estatísticas no domínio de campo.
Nenhum peso é treinado.

In [ ]:
def adabn(model, x, batch=64):
    bns = bn_layers(model)
    orig = [l.momentum for l in bns]
    ds = make_ds(x, np.zeros(len(x), np.int32), training=False, batch=batch)
    for k, (xb, _) in enumerate(ds, start=1):
        for l in bns:
            l.momentum = (k - 1) / k          # média acumulada: no fim, cada lote pesa igual
        model(xb, training=True)
    for l, m in zip(bns, orig):
        l.momentum = m
    return model

def adabn_variant(run, src_path, variant):
    rdir = f'{OUT_DIR}/{run}_{variant.replace("+", "_")}'
    os.makedirs(rdir, exist_ok=True)
    keras.backend.clear_session()
    model = keras.models.load_model(src_path, compile=False)
    model = adabn(model, x_pdtr)
    model.save(f'{rdir}/model.keras')
    return save_eval(model, run, variant, rdir)

## 7. Treino e fine-tuning

In [ ]:
class EpochTime(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None): logs['epoch_time_s'] = time.time() - self.t0

def fit(model, ds_train, ds_val, epochs, lr, steps, rdir, patience):
    sched = keras.optimizers.schedules.CosineDecay(0.0, epochs * steps, warmup_target=lr, warmup_steps=steps)
    model.compile(optimizer=keras.optimizers.AdamW(sched, weight_decay=WD),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
                  metrics=['accuracy'])
    if not os.path.exists(f'{rdir}/train_done'):
        model.fit(ds_train, validation_data=ds_val, epochs=epochs, verbose=2, callbacks=[
            keras.callbacks.BackupAndRestore(f'{rdir}/backup'),
            EpochTime(),
            keras.callbacks.ModelCheckpoint(f'{rdir}/best.keras', monitor='val_accuracy', save_best_only=True),
            keras.callbacks.CSVLogger(f'{rdir}/history.csv', append=True),
            keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=patience),
        ])
        open(f'{rdir}/train_done', 'w').close()
    return keras.models.load_model(f'{rdir}/best.keras', compile=False)

def train_aug(run):
    # do zero, só PlantVillage, aumento forte. Seleção da época pela validação do PlantVillage.
    rdir = f'{OUT_DIR}/{run}_aug'; os.makedirs(rdir, exist_ok=True)
    if os.path.exists(f'{rdir}/best.keras') and os.path.exists(f'{rdir}/train_done'):
        return keras.models.load_model(f'{rdir}/best.keras', compile=False)
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    ds_tr = make_ds(x_tr, y_tr, True, BATCH, augment_strong)
    ds_va = make_ds(x_va, y_va, False, 256)
    print(f'== {run}: treino do zero com aumento forte')
    return fit(build_model(run), ds_tr, ds_va, EPOCHS_AUG, LR, math.ceil(len(y_tr) / BATCH), rdir, PATIENCE)

def train_ft(run):
    # parte do modelo 'aug'; treina com campo (85% do treino do PlantDoc) + PlantVillage.
    # Seleção da época pelos 15% de validação do PlantDoc (o teste oficial não é usado).
    rdir = f'{OUT_DIR}/{run}_aug_ft'; os.makedirs(rdir, exist_ok=True)
    if os.path.exists(f'{rdir}/best.keras') and os.path.exists(f'{rdir}/train_done'):
        return keras.models.load_model(f'{rdir}/best.keras', compile=False)
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    model = keras.models.load_model(f'{OUT_DIR}/{run}_aug/best.keras', compile=False)
    ds_tr = make_ds(x_ft, y_ft, True, BATCH_FT, augment_strong)
    ds_va = make_ds(x_ftva, y_ftva, False, 256)
    print(f'== {run}: fine-tuning com fotos de campo')
    return fit(model, ds_tr, ds_va, EPOCHS_FT, LR_FT, math.ceil(len(y_ft) / BATCH_FT), rdir, PATIENCE_FT)

## 8. Execução (todos os modelos e variantes)

In [ ]:
results = []
for run in RUNS:
    t0 = time.time()
    rdir_base = f'{OUT_DIR}/{run}_base_eval'; os.makedirs(rdir_base, exist_ok=True)
    keras.backend.clear_session()
    results.append(save_eval(keras.models.load_model(f'{BASE_DIR}/{run}/best.keras', compile=False),
                             run, 'base', rdir_base))
    results.append(adabn_variant(run, f'{BASE_DIR}/{run}/best.keras', 'base+adabn'))

    m_aug = train_aug(run)
    results.append(save_eval(m_aug, run, 'aug', f'{OUT_DIR}/{run}_aug'))
    results.append(adabn_variant(run, f'{OUT_DIR}/{run}_aug/best.keras', 'aug+adabn'))

    m_ft = train_ft(run)
    results.append(save_eval(m_ft, run, 'aug+ft', f'{OUT_DIR}/{run}_aug_ft'))
    print(f'[{run}] concluído em {(time.time() - t0) / 60:.1f} min\n')

## 9. Tabela e figura

In [ ]:
NAMES = {'cnn_base': 'CNN própria', 'mnv3s': 'MobileNetV3-Small', 'mnv2_050': 'MobileNetV2 (α=0.5)'}
ORDER = ['base', 'base+adabn', 'aug', 'aug+adabn', 'aug+ft']
df = pd.DataFrame(results)
df['modelo'] = df['run'].map(NAMES)
df['variante'] = pd.Categorical(df['variante'], ORDER, ordered=True)
df = df.sort_values(['run', 'variante'])
cols = ['modelo', 'variante', 'pv_acc', 'pd_acc', 'pd_f1', 'pd_acc_restrito', 'pd_acc_cultura',
        'pd_ece', 'pd_cob@0.7', 'pd_acc@0.7']
df = df[cols]

if not QUICK:
    df.to_csv(f'{TAB_DIR}/tab_mitigacao.csv', index=False)
    tex = df.copy()
    for c in cols[2:]:
        tex[c] = (100 * tex[c]).round(1) if c != 'pd_ece' else tex[c].round(3)
    tex.columns = ['Modelo', 'Variante', 'Acc PV', 'Acc PD', 'F1 PD', 'Acc PD (restr.)', 'Acc cultura PD',
                   'ECE PD', 'Cob. PD@0.7', 'Acc PD@0.7']
    tex.to_latex(f'{TAB_DIR}/tab_mitigacao.tex', index=False, float_format='%.1f')

fig, ax = plt.subplots(figsize=(9, 4))
w = 0.27
for j, (run, g) in enumerate(df.groupby('modelo', sort=False)):
    ax.bar(np.arange(len(ORDER)) + (j - 1) * w, 100 * g['pd_acc'].values, w, label=run)
ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER)
ax.set_ylabel('Acurácia no PlantDoc (teste, %)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
if not QUICK:
    plt.savefig(f'{FIG_DIR}/mitigacao.png', dpi=200, bbox_inches='tight')
plt.show()

print(df.to_string(index=False, float_format='%.3f'))